In [1]:
import pandas as pd

file_name = "translation_result_of_fine-tuned-models_nllb-200-distilled-600M_zh2ko_1031.xlsx"
df = pd.read_excel(file_name)
df

,source,references,candidates
0,2天22<start>小时<middle>시간<end>22<start>分钟<middle...,2일 22<start>시간<end> 22<start>분<end>,2일 22 시간 22 분
1,"{阿登丘陵的亚岱尔,找亚岱尔<start>交谈<middle>대화<end>,序章}","{아르덴 산맥의 아데어,아데어와 <start>대화<end>하기,프롤로그}","{아르덴 산맥의 아데어,아데어와 대화 하기,프롤로그}"
2,<start><start>神<middle>신<end>秘<middle>신비<end>宝...,<start><start>신<end>비<end><start>상자<end>,신 비 한 보물 상자
3,骑战装备熔炉<start>等级达到<middle>레벨 달성<end>4级,기마전투 장비 용로 4<start>레벨 달성<end>,기마전투 장비 용로 4 레벨 달성
4,<start>菜鸟<middle>신입<end>玉镯,<start>신입<end> 옥팔찌,신입 옥팔찌
...,...,...,...
15497,受内攻伤害削减<c=green>1.5%</c>（内攻伤害包括：气功伤害、内力）,"받는 내공피해 <c=green>1.5%</c> 감소(내공피해는 기공피해, 내력)","받는 내공피해 <c=green> 1.5% </c> 감소(내공피해는 기공피해, 내력)"
15498,<c=yellow>双生雀</c>：谢谢浣熊大师教导，我记住了。,<c=yellow>참새부부</c>\n라쿤사부님의 가르침 감사합니다.,<c=yellow> 참새부부부 </c> \n라답대사에게 가르침을 받았네. 기억하네.
15499,"{魔力之书（精通闪避Ⅱ）,使用后学会通用技能【精通闪避Ⅱ】,<start>制作<middle...","{지식의 서 (회피 마스터리 Ⅱ),사용 후, 공통의 스킬 【회피 마스터리 Ⅱ】를 습...","{지식의 서 (회피 마스터리 II),사용 후, 공통의 스킬 【 회피 마스터리 II】..."
15500,正在<start>收集<middle>수집<end>{0},현재 {0} <start>수집<end>,{0} 수집


In [2]:
import re

for index, row in df.iterrows():
    text_source = str(row["source"])
    # code special tags
    suffix_tags = list(
        set(re.findall("|".join(["\[\/.*?\]", "\<\/.*?\>"]), text_source)))  # e.g. ['[/color]', '[/size]']
    suffix_tag_types = [re.search("\w+", tag).group() for tag in suffix_tags]  # e.g.['size', 'color']
    regex = "|".join(
        [r"\[{0}=.*?\]".format(tag_type) for tag_type in suffix_tag_types] +
        [r"\<{0}=.*?\>".format(tag_type) for tag_type in suffix_tag_types])  # e.g.'\[size=.*?\]|\[color=.*?\]'
    prefix_tags = list(set(re.findall(regex, text_source) if regex else []))  # e.g. ['[color=#F7C358]', '[size=40]']
    tags_all_source = suffix_tags + prefix_tags  # e.g. ['[/color]', '[/size]', '[color=#F7C358]', '[size=40]']

    text_candidates = str(row["candidates"])
    # code special tags
    suffix_tags = list(
        set(re.findall("|".join(["\[\/.*?\]", "\<\/.*?\>"]), text_candidates)))  # e.g. ['[/color]', '[/size]']
    suffix_tag_types = [re.search("\w+", tag).group() for tag in suffix_tags]  # e.g.['size', 'color']
    regex = "|".join(
        [r"\[{0}=.*?\]".format(tag_type) for tag_type in suffix_tag_types] +
        [r"\<{0}=.*?\>".format(tag_type) for tag_type in suffix_tag_types])  # e.g.'\[size=.*?\]|\[color=.*?\]'
    prefix_tags = list(
        set(re.findall(regex, text_candidates) if regex else []))  # e.g. ['[color=#F7C358]', '[size=40]']
    tags_all_candidates = suffix_tags + prefix_tags  # e.g. ['[/color]', '[/size]', '[color=#F7C358]', '[size=40]']

    if len(tags_all_source) == 0:
        df.at[index, "matched"] = 2
    elif set(tags_all_source) == set(tags_all_candidates):
        df.at[index, "matched"] = 1
    elif set(tags_all_source) != set(tags_all_candidates):
        df.at[index, "matched"] = 0

In [3]:
df.to_excel(file_name, index=False)
df

,source,references,candidates,matched
0,2天22<start>小时<middle>시간<end>22<start>分钟<middle...,2일 22<start>시간<end> 22<start>분<end>,2일 22 시간 22 분,2.0
1,"{阿登丘陵的亚岱尔,找亚岱尔<start>交谈<middle>대화<end>,序章}","{아르덴 산맥의 아데어,아데어와 <start>대화<end>하기,프롤로그}","{아르덴 산맥의 아데어,아데어와 대화 하기,프롤로그}",2.0
2,<start><start>神<middle>신<end>秘<middle>신비<end>宝...,<start><start>신<end>비<end><start>상자<end>,신 비 한 보물 상자,2.0
3,骑战装备熔炉<start>等级达到<middle>레벨 달성<end>4级,기마전투 장비 용로 4<start>레벨 달성<end>,기마전투 장비 용로 4 레벨 달성,2.0
4,<start>菜鸟<middle>신입<end>玉镯,<start>신입<end> 옥팔찌,신입 옥팔찌,2.0
...,...,...,...,...
15497,受内攻伤害削减<c=green>1.5%</c>（内攻伤害包括：气功伤害、内力）,"받는 내공피해 <c=green>1.5%</c> 감소(내공피해는 기공피해, 내력)","받는 내공피해 <c=green> 1.5% </c> 감소(내공피해는 기공피해, 내력)",1.0
15498,<c=yellow>双生雀</c>：谢谢浣熊大师教导，我记住了。,<c=yellow>참새부부</c>\n라쿤사부님의 가르침 감사합니다.,<c=yellow> 참새부부부 </c> \n라답대사에게 가르침을 받았네. 기억하네.,1.0
15499,"{魔力之书（精通闪避Ⅱ）,使用后学会通用技能【精通闪避Ⅱ】,<start>制作<middle...","{지식의 서 (회피 마스터리 Ⅱ),사용 후, 공통의 스킬 【회피 마스터리 Ⅱ】를 습...","{지식의 서 (회피 마스터리 II),사용 후, 공통의 스킬 【 회피 마스터리 II】...",2.0
15500,正在<start>收集<middle>수집<end>{0},현재 {0} <start>수집<end>,{0} 수집,2.0
